# Auditoría de tipos de datos sobre el archivo completo

Comprobamos si podemos reducir memoria **sin cambiar valores**, para ello recorreremos todas columnas del CSV en bloques de 50 000 filas.

Queremos obtener resultados para los siguientes métodos: la forma **conservadora**, con enteros más pequeños y categorías cuando la conversión es exacta y **experimental**, que además convierte los decimales a `float32`. 

La referencia es la representación que usa actualmente el proyecto (`float64` para columnas numéricas y `string` para texto). Comprobaremos pérdidas o variaciones adicionales respecto a esa lectura. Las fechas y el plazo se mantienen como texto en esta auditoría para aislar el cambio de almacenamiento.

In [2]:
from pathlib import Path
import sys
import time
import platform
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
assert (RAIZ / "src").exists(), "Ejecutar desde el proyecto o desde notebooks."
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src.data import inferir_tipos, COLUMNAS_FECHA
from src.config import localizar_datos

archivo_local = RAIZ / "accepted_2007_to_2018Q4.csv"
archivo = archivo_local if archivo_local.exists() else localizar_datos()
TAMANO_BLOQUE = 50_000
MAX_CATEGORIAS = 1_000
inicio = time.perf_counter()
tipos, columnas = inferir_tipos(archivo)
numericas = [c for c in columnas if tipos[c] == "float64"]
texto = [c for c in columnas if tipos[c] == "string"]
print("Archivo:", archivo.name)
print("Columnas:", len(columnas))
print("Versiones:", platform.python_version(), "pandas", pd.__version__, "numpy", np.__version__)

Archivo: accepted_2007_to_2018Q4.csv
Columnas: 151
Versiones: 3.12.10 pandas 3.0.3 numpy 2.4.6


## 1. Revisar todos los rangos y categorías

La decisión de reducir tipos usa **el archivo completo**, de modo que compararemos el cambio en los mínimos, máximos, presencia de decimales, infinitos y faltantes. Si una columna tiene muchos textos diferentes, dejamos de acumular su vocabulario y conservamos su tipo original. El límite de 1 000 categorías evita guardar listas enormes.

In [3]:
perfil = {c: {"observados": 0, "faltantes": 0, "minimo": np.inf,
              "maximo": -np.inf, "todos_enteros": True, "no_finitos": 0}
          for c in numericas}
vocabularios = {c: set() for c in texto if c not in COLUMNAS_FECHA}
memoria_original = pd.Series(0, index=columnas, dtype="int64")
filas = 0
bloques = 0
with pd.read_csv(archivo, dtype=tipos, chunksize=TAMANO_BLOQUE,
                 skipinitialspace=True) as lector:
    for bloque in lector:
        filas += len(bloque)
        bloques += 1
        memoria_original += bloque.memory_usage(index=False, deep=True)
        for nombre in numericas:
            serie = bloque[nombre]
            valores = serie.dropna().to_numpy()
            p = perfil[nombre]
            p["observados"] += len(valores)
            p["faltantes"] += int(serie.isna().sum())
            finitos = valores[np.isfinite(valores)]
            p["no_finitos"] += int((~np.isfinite(valores)).sum())
            if len(finitos):
                p["minimo"] = min(p["minimo"], float(finitos.min()))
                p["maximo"] = max(p["maximo"], float(finitos.max()))
                p["todos_enteros"] &= bool((finitos == np.trunc(finitos)).all())
        for nombre, valores in vocabularios.items():
            if valores is not None:
                valores.update(bloque[nombre].dropna().unique().tolist())
                if len(valores) > MAX_CATEGORIAS:
                    vocabularios[nombre] = None

print(f"Archivo completo: {filas:,} filas, {len(columnas)} columnas y {bloques} bloques.")
print("No se excluyó ningún estado de préstamo.")

Archivo completo: 2,260,701 filas, 151 columnas y 46 bloques.
No se excluyó ningún estado de préstamo.


## 2. Elegir tipos que puedan conservar los valores

Para números enteros usamos `Int8`, `Int16` o `Int32`, según el rango completo. Conservamos `float64` si hay decimales, infinitos, una columna vacía o no existe una reducción segura entre esos tipos.

Para texto repetido usamos un vocabulario común a todos los bloques. Después verificaremos que ocupa menos memoria y que recupera exactamente cada texto. No sustituimos las categorías por un orden numérico para el modelo.

In [4]:
tipos_seguros = dict(tipos)
for nombre, p in perfil.items():
    if p["observados"] and p["todos_enteros"] and p["no_finitos"] == 0:
        for tipo in ["Int8", "Int16", "Int32"]:
            limites = np.iinfo(tipo.lower())
            if limites.min <= p["minimo"] and p["maximo"] <= limites.max:
                tipos_seguros[nombre] = tipo
                break
for nombre, valores in vocabularios.items():
    if valores:
        tipos_seguros[nombre] = pd.CategoricalDtype(categories=sorted(valores))

resumen_rangos = pd.DataFrame(perfil).T
resumen_rangos[["minimo", "maximo"]] = resumen_rangos[["minimo", "maximo"]].replace([np.inf, -np.inf], np.nan)
resumen_rangos["tipo_original"] = "float64"
resumen_rangos["tipo_propuesto"] = [str(tipos_seguros[c]) for c in resumen_rangos.index]
with pd.option_context("display.max_rows", None):
    display(resumen_rangos)

,observados,faltantes,minimo,maximo,todos_enteros,no_finitos,tipo_original,tipo_propuesto
loan_amnt,2260668,33,500.0,40000.0,True,0,float64,Int32
funded_amnt,2260668,33,500.0,40000.0,True,0,float64,Int32
funded_amnt_inv,2260668,33,0.0,40000.0,False,0,float64,float64
int_rate,2260668,33,5.31,30.99,False,0,float64,float64
installment,2260668,33,4.93,1719.83,False,0,float64,float64
annual_inc,2260664,37,0.0,110000000.0,False,0,float64,float64
dti,2258957,1744,-1.0,999.0,False,0,float64,float64
delinq_2yrs,2260639,62,0.0,58.0,True,0,float64,Int8
fico_range_low,2260668,33,610.0,845.0,True,0,float64,Int16
fico_range_high,2260668,33,614.0,850.0,True,0,float64,Int16


## 3. Convertir y comprobar todas las filas

Volvemos a leer el archivo completo y convertimos cada bloque. Comparamos los faltantes y los valores mediante ida y vuelta al tipo original. Un cambio numérico cuenta aunque sea muy pequeño.

En el experimento `float32` registramos cuántos valores cambian, el mayor error absoluto, el mayor error relativo y si aparecen infinitos o ceros por pérdida de rango. Esta opción no se considera exacta por el mero hecho de que el error sea pequeño.

In [5]:
memoria_segura = pd.Series(0, index=columnas, dtype="int64")
memoria_float32 = pd.Series(0, index=columnas, dtype="int64")
verificacion = {c: {"valores_cambiados": 0, "faltantes_cambiados": 0} for c in columnas}
prueba_float32 = {c: {"observados": 0, "valores_cambiados": 0,
                     "error_absoluto_max": 0.0, "error_relativo_max": 0.0,
                     "nuevos_infinitos": 0, "nuevos_ceros": 0}
                 for c in numericas if tipos_seguros[c] == "float64"}
filas_verificadas = 0
with pd.read_csv(archivo, dtype=tipos, chunksize=TAMANO_BLOQUE,
                 skipinitialspace=True) as lector:
    for bloque in lector:
        filas_verificadas += len(bloque)
        for nombre in columnas:
            original = bloque[nombre]
            convertida = original.astype(tipos_seguros[nombre])
            memoria_segura[nombre] += convertida.memory_usage(index=False, deep=True)
            vuelta = convertida.astype(tipos[nombre])
            verificacion[nombre]["faltantes_cambiados"] += int((original.isna() != vuelta.isna()).sum())
            observados = original.notna()
            verificacion[nombre]["valores_cambiados"] += int(
                (original[observados] != vuelta[observados]).fillna(True).sum())
            experimental = convertida
            if nombre in prueba_float32:
                with np.errstate(over="ignore", under="ignore", invalid="ignore"):
                    experimental = original.astype("float32")
                a = original[observados].to_numpy()
                b = experimental[observados].to_numpy(dtype="float64")
                r = prueba_float32[nombre]
                r["observados"] += len(a)
                r["valores_cambiados"] += int((a != b).sum())
                finitos = np.isfinite(a)
                r["nuevos_infinitos"] += int((finitos & np.isinf(b)).sum())
                r["nuevos_ceros"] += int((finitos & (a != 0) & (b == 0)).sum())
                if finitos.any():
                    errores = np.abs(a[finitos] - b[finitos])
                    r["error_absoluto_max"] = max(r["error_absoluto_max"], float(errores.max()))
                    no_cero = a[finitos] != 0
                    if no_cero.any():
                        relativos = errores[no_cero] / np.abs(a[finitos][no_cero])
                        r["error_relativo_max"] = max(r["error_relativo_max"], float(relativos.max()))
            memoria_float32[nombre] += experimental.memory_usage(index=False, deep=True)

assert filas_verificadas == filas
comprobacion = pd.DataFrame(verificacion).T
assert comprobacion.to_numpy().sum() == 0, "Se detectó una pérdida en la propuesta conservadora."
# Si una categoría no ahorra memoria, mantenemos el texto original.
for nombre in columnas:
    if memoria_segura[nombre] >= memoria_original[nombre]:
        tipos_seguros[nombre] = tipos[nombre]
        memoria_segura[nombre] = memoria_original[nombre]
        if nombre not in prueba_float32:
            memoria_float32[nombre] = memoria_original[nombre]
print(f"Verificadas {filas_verificadas:,} filas y {len(columnas)} columnas.")
print("Cambios de valores en la opción conservadora:", int(comprobacion.valores_cambiados.sum()))
print("Cambios de faltantes en la opción conservadora:", int(comprobacion.faltantes_cambiados.sum()))

Verificadas 2,260,701 filas y 151 columnas.
Cambios de valores en la opción conservadora: 0
Cambios de faltantes en la opción conservadora: 0


## 4. ¿Qué sucede con float32?

Un cambio de `float64` a `float32` puede redondear un número aunque no haya desbordamiento. Revisamos el error por columna; no establecemos una tolerancia universal ni asumimos que las métricas del modelo serán idénticas.

In [6]:
errores_float32 = pd.DataFrame(prueba_float32).T
errores_float32["porcentaje_cambiado"] = 100 * errores_float32.valores_cambiados / errores_float32.observados.replace(0, np.nan)

with pd.option_context("display.max_rows", None, "display.float_format", "{:.8g}".format):
    display(errores_float32.sort_values("error_absoluto_max", ascending=False))
print("Valores alterados por float32:", int(errores_float32.valores_cambiados.sum()))
print("Infinitos nuevos:", int(errores_float32.nuevos_infinitos.sum()))
print("Ceros nuevos:", int(errores_float32.nuevos_ceros.sum()))

,observados,valores_cambiados,error_absoluto_max,error_relativo_max,nuevos_infinitos,nuevos_ceros,porcentaje_cambiado
annual_inc,2260664,16099,0.04,5.7207505e-08,0,0,0.7121359
annual_inc_joint,120710,1447,0.015,5.7199372e-08,0,0,1.1987408
total_pymnt,2260668,2204488,0.0019531126,5.9544508e-08,0,0,97.514894
funded_amnt_inv,2260668,8449,0.001914017,5.7518127e-08,0,0,0.37373909
total_rec_prncp,2260668,1128801,0.001875,5.7220319e-08,0,0,49.932188
out_prncp,2260668,870382,0.001875,5.7218853e-08,0,0,38.501098
total_pymnt_inv,2260668,2167159,0.001875,5.7220319e-08,0,0,95.863656
out_prncp_inv,2260668,870515,0.001875,5.7218853e-08,0,0,38.506981
last_pymnt_amnt,2260668,2140451,0.001875,5.7586805e-08,0,0,94.682236
hardship_payoff_balance_amount,10917,10481,0.001875,5.7098344e-08,0,0,96.006229


Valores alterados por float32: 23412405
Infinitos nuevos: 0
Ceros nuevos: 0


## 5. Resultado de memoria y tipos recomendados

Sumamos la memoria profunda de cada columna en cada bloque, **sin índices**. Los vocabularios categóricos se contabilizan en cada bloque, por lo que la suma incluye su coste repetido. Las tres alternativas se comparan con el mismo criterio.

In [7]:
detalle = pd.DataFrame({
    "tipo_original": pd.Series(tipos),
    "tipo_conservador": pd.Series({c: str(tipos_seguros[c]) for c in columnas}),
    "original_MiB": memoria_original / 1024**2,
    "conservador_MiB": memoria_segura / 1024**2,
    "experimental_MiB": memoria_float32 / 1024**2,
}).join(comprobacion)
detalle["ahorro_conservador_MiB"] = detalle.original_MiB - detalle.conservador_MiB
resumen = pd.DataFrame({"memoria_acumulada_MiB": [memoria_original.sum()/1024**2,
    memoria_segura.sum()/1024**2, memoria_float32.sum()/1024**2]},
    index=["Lectura actual", "Conservadora (exacta)", "Experimental (+ float32)"])
resumen["reduccion_pct"] = 100 * (1 - resumen.memoria_acumulada_MiB / resumen.iloc[0, 0])
display(resumen.round(2))
print("Tipos finales propuestos:")
display(detalle.tipo_conservador.value_counts().to_frame("columnas"))
with pd.option_context("display.max_rows", None):
    display(detalle.sort_values("ahorro_conservador_MiB", ascending=False).round(4))

,memoria_acumulada_MiB,reduccion_pct
Lectura actual,6887.47,0.00
Conservadora (exacta),2360.93,65.72
Experimental (+ float32),2119.46,69.23


Tipos finales propuestos:


,columnas
tipo_conservador,
Int8,43
category,31
float64,28
Int16,24
Int32,16
string,9


,tipo_original,tipo_conservador,original_MiB,conservador_MiB,experimental_MiB,valores_cambiados,faltantes_cambiados,ahorro_conservador_MiB
purpose,string,category,137.5346,2.1923,2.1923,0,0,135.3423
verification_status,string,category,131.6478,2.1640,2.1640,0,0,129.4839
application_type,string,category,127.0872,2.1611,2.1611,0,0,124.9261
loan_status,string,category,125.2840,2.1835,2.1835,0,0,123.1006
term,string,category,125.0463,2.1611,2.1611,0,0,122.8853
emp_length,string,category,122.2008,2.1831,2.1831,0,0,120.0177
next_pymnt_d,string,category,121.6074,2.4210,2.4210,0,0,119.1864
verification_status_joint,string,category,121.2963,2.1640,2.1640,0,0,119.1323
hardship_type,string,category,120.9843,2.1595,2.1595,0,0,118.8248
hardship_loan_status,string,category,120.8069,2.1695,2.1695,0,0,118.6374


## 6. Conclusión de la ejecución y límites

La comprobación exacta protege frente a truncamiento, pérdida de faltantes o sustitución de categorías en **este archivo**. Los tipos se eligieron a partir de sus rangos completos.

El ahorro de almacenamiento no garantiza un ahorro igual al entrenar: el preprocesamiento actual convierte texto a `object`, genera matrices densas y algunos algoritmos convierten números nuevamente a 64 bits. Hasta esta sección solo auditamos la representación. Desde la sección 8 comparamos únicamente el tiempo de carga directa y por bloques. No medimos RAM ni entrenamos modelos en esta comparación.

In [8]:
print(f"Filas auditadas: {filas:,}; columnas: {len(columnas)}.")
print(f"Reducción conservadora acumulada: {resumen.loc['Conservadora (exacta)', 'reduccion_pct']:.2f}%.")
print("La opción conservadora preservó exactamente valores y faltantes respecto de la lectura actual.")
if errores_float32.valores_cambiados.sum():
    print("float32 NO conserva exactamente todos los valores: mantener float64 en esas columnas si se exige igualdad exacta.")
else:
    print("float32 no produjo cambios observables en este archivo bajo esta comparación.")


Filas auditadas: 2,260,701; columnas: 151.
Reducción conservadora acumulada: 65.72%.
La opción conservadora preservó exactamente valores y faltantes respecto de la lectura actual.
float32 NO conserva exactamente todos los valores: mantener float64 en esas columnas si se exige igualdad exacta.


## 7. Preparar la comparación de carga

Compararemos cuánto tarda en quedar disponible el mismo conjunto de préstamos con dos métodos. En ambos casos leeremos las mismas columnas seleccionadas, usaremos los tipos conservadores auditados y conservaremos los mismos estados finales (`Fully Paid` y `Charged Off`).

- **Lectura directa:** Pandas lee de una vez todas las filas de las columnas seleccionadas y después filtra los préstamos elegibles.
- **Lectura por bloques:** Pandas lee 50 000 filas cada vez, filtra cada bloque y junta al final los préstamos elegibles.

El tiempo incluye lectura, filtro y unión de bloques cuando corresponda. No incluye la auditoría de tipos, el cálculo de huellas, la preparación para el modelo ni el entrenamiento. La lectura por bloques también termina reuniendo los préstamos elegibles en memoria.

In [9]:
import json
import subprocess
import tempfile
from pathlib import Path

from IPython.display import display
from src.features import seleccionar_predictoras

SALIDA_COMPARACION = RAIZ / "outputs" / "comparacion_carga"
SALIDA_COMPARACION.mkdir(parents=True, exist_ok=True)

candidatas = seleccionar_predictoras(pd.DataFrame(columns=columnas), [], False)
columnas_comparacion = [
    columna for columna in columnas
    if columna in candidatas + ["loan_status", "issue_d"]
]
esquema_comparacion = {}
for nombre in columnas_comparacion:
    tipo = tipos_seguros[nombre]
    if isinstance(tipo, pd.CategoricalDtype):
        esquema_comparacion[nombre] = {
            "tipo": "category",
            "categorias": tipo.categories.tolist(),
        }
    else:
        esquema_comparacion[nombre] = {"tipo": str(tipo)}

config_comparacion = {
    "archivo": str(archivo),
    "esquema": esquema_comparacion,
    "filas_fuente_esperadas": filas,
    "bloque": TAMANO_BLOQUE,
}

print("Columnas leídas en ambos métodos:", len(columnas_comparacion))
print("Filas fuente que leerá cada ejecución:", f"{filas:,}")
print("Tamaño de bloque para el segundo método:", f"{TAMANO_BLOQUE:,} filas")

Columnas leídas en ambos métodos: 97
Filas fuente que leerá cada ejecución: 2,260,701
Tamaño de bloque para el segundo método: 50,000 filas


## 8. Ejecutar las mediciones de tiempo

Cada ejecución se hace en un proceso nuevo. Alternamos el orden para reducir el efecto de ejecutar siempre un método primero. Hacemos tres repeticiones por método; los resultados siguen siendo propios de este equipo y de estas condiciones de lectura.

In [10]:
def ejecutar_prueba_carga(metodo, repeticion, carpeta_temporal):
    carpeta = carpeta_temporal / f"{metodo}_{repeticion}"
    carpeta.mkdir()
    ruta_config = carpeta / "config.json"
    ruta_config.write_text(json.dumps(config_comparacion), encoding="utf-8")
    ruta_worker = RAIZ / "src" / "benchmark_worker.py"
    assert ruta_worker.exists(), f"No se encontró el programa de medición: {ruta_worker}"

    proceso = subprocess.run(
        [sys.executable, str(ruta_worker), str(ruta_config), metodo, str(carpeta)],
        cwd=RAIZ,
        capture_output=True,
        text=True,
        encoding="utf-8",
    )
    if proceso.returncode != 0:
        raise RuntimeError(
            f"Falló la ejecución {metodo} {repeticion}:\n{proceso.stderr[-4000:]}"
        )
    print(proceso.stdout.strip())
    resultado = json.loads((carpeta / "resultado.json").read_text(encoding="utf-8"))
    resultado["repeticion"] = repeticion
    destino = SALIDA_COMPARACION / f"{metodo}_{repeticion}.json"
    destino.write_text(json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8")
    return resultado

orden_pruebas = [
    ("directa", 1), ("bloques", 1),
    ("bloques", 2), ("directa", 2),
    ("directa", 3), ("bloques", 3),
]
resultados_carga = []
with tempfile.TemporaryDirectory(prefix="comparacion_carga_") as temporal:
    carpeta_temporal = Path(temporal)
    for metodo, repeticion in orden_pruebas:
        print(f"\nMidiendo {metodo}, repetición {repeticion}...")
        resultados_carga.append(
            ejecutar_prueba_carga(metodo, repeticion, carpeta_temporal)
        )
print("\nMediciones terminadas.")


Midiendo directa, repetición 1...
directa: carga y filtro en 41.74 s

Midiendo bloques, repetición 1...
bloques: carga y filtro en 61.69 s

Midiendo bloques, repetición 2...
bloques: carga y filtro en 41.81 s

Midiendo directa, repetición 2...
directa: carga y filtro en 56.63 s

Midiendo directa, repetición 3...
directa: carga y filtro en 42.34 s

Midiendo bloques, repetición 3...
bloques: carga y filtro en 53.71 s

Mediciones terminadas.


## 9. Resultados y comprobación de igualdad

Primero verificamos que ambas opciones recorrieron las mismas filas y produjeron el mismo conjunto de préstamos. Después comparamos los tiempos individuales y la mediana de cada método.

In [11]:
tabla_tiempos = pd.DataFrame([
    {
        "método": resultado["metodo"],
        "repetición": resultado["repeticion"],
        "filas_leídas": resultado["filas_fuente"],
        "préstamos_elegibles": resultado["filas_elegibles"],
        "tiempo_segundos": resultado["segundos_carga"],
        "huella": resultado["huella_datos"],
    }
    for resultado in resultados_carga
])

assert tabla_tiempos["filas_leídas"].nunique() == 1
assert tabla_tiempos["préstamos_elegibles"].nunique() == 1
assert tabla_tiempos["huella"].nunique() == 1
print("Las dos formas leyeron y conservaron los mismos datos.")
display(tabla_tiempos.drop(columns="huella").round(2))

resumen_tiempos = tabla_tiempos.groupby("método")["tiempo_segundos"].agg(
    repeticiones="count",
    mediana_segundos="median",
    promedio_segundos="mean",
    mínimo_segundos="min",
    máximo_segundos="max",
).sort_values("mediana_segundos")
display(resumen_tiempos.round(2))

más_rápido = resumen_tiempos.index[0]
menos_rápido = resumen_tiempos.index[-1]
mediana_rápido = resumen_tiempos.iloc[0]["mediana_segundos"]
mediana_lento = resumen_tiempos.iloc[-1]["mediana_segundos"]
porcentaje = (mediana_lento - mediana_rápido) / mediana_lento * 100
print(f"Método más rápido por mediana: {más_rápido}.")
print(f"Su mediana fue {porcentaje:.1f}% menor que la del otro método.")

Las dos formas leyeron y conservaron los mismos datos.


,método,repetición,filas_leídas,préstamos_elegibles,tiempo_segundos
0,directa,1,2260701,1345310,41.74
1,bloques,1,2260701,1345310,61.69
2,bloques,2,2260701,1345310,41.81
3,directa,2,2260701,1345310,56.63
4,directa,3,2260701,1345310,42.34
5,bloques,3,2260701,1345310,53.71


,repeticiones,mediana_segundos,promedio_segundos,mínimo_segundos,máximo_segundos
método,,,,,
directa,3,42.34,46.9,41.74,56.63
bloques,3,53.71,52.4,41.81,61.69


Método más rápido por mediana: directa.
Su mediana fue 21.2% menor que la del otro método.


## 10. Referencias

- Python Software Foundation. (s. f.). *time.perf_counter*. https://docs.python.org/3/library/time.html#time.perf_counter
- The pandas development team. (s. f.). *pandas.read_csv*. https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
- The pandas development team. (s. f.). *Scaling to large datasets*. https://pandas.pydata.org/docs/user_guide/scale.html